In [1]:
import pandas as pd
import numpy as np
from plots import plot_appendix, plot_main_paper
from kcit import kci_test
import pickle

In [9]:
RESULTS_DIR = 'kcit_randomized_graph_results'

# Simulation 6 Experiments

In [10]:
def clamp(s):
  """Limits a value to be within the range [0, 1]."""
  return s.clip(lower=0, upper=1)


def generate_sgc_datasets(n,
                          true_upcoding_rates: list[float], true_downcoding_rates: list[float],
                          genuine_modifications_a_1_on_xstar: list[float], genuine_modifications_a_0_on_xstar: list[float]):
    """
    n = number of features (X*'s)
    assuming just 1 downstream variable Y for now, I believe this is WLOG
    all other args are lists of values, 1 list entry per feature (X*) """

    # Load dataset
    df = pd.read_excel('./datasets/default_of_credit_card_clients.xls', skiprows=1)
    df = df.sample(frac=1).reset_index(drop=True)

    # Drop rows that are not needed
    df = df[['SEX', 'EDUCATION', 'MARRIAGE', 'AGE']]

    # Convert sex to binary variable
    df['SEX'] = df['SEX'] - 1

    # Convert marriage to binary variable
    df['MARRIAGE'] = np.where(df['MARRIAGE'] > 1, 1, 0)

    # Convert education to binary variable
    df['EDUCATION'] = np.where(df['EDUCATION'] > 2, 1, 0)

    # Min-max scale the data
    df['AGE'] = (df['AGE'] - df['AGE'].min()) / (df['AGE'].max() - df['AGE'].min())
    
    # Keeping it simple by including no selection bias in agent_prob for now
    agent_prob = 0.3
    df['AGENT'] = np.random.binomial(1, agent_prob, len(df))

    causal_effect_of_x_on_y_list = []
    y_prob = 0.05 \
                    + df['EDUCATION']*0.05 \
                    + df['MARRIAGE']*df['SEX']*0.3 \
                    + np.square(df['AGE'])*0.1
    for i in range(n):
        variable_name = f"X{i}"

        x_prob = 0.05 \
                        + df['EDUCATION']*0.05 \
                        + df['MARRIAGE']*df['SEX']*0.3 \
                        + np.square(df['AGE'])*0.1 \
                        + df['AGENT'] * genuine_modifications_a_1_on_xstar[i] + (1-df['AGENT']) * genuine_modifications_a_0_on_xstar[i]
        df[variable_name] = np.random.binomial(1, clamp(x_prob), len(df))

        # Strategically misreport the dataset (misreported employment status)
        prob_required_for_upcoding_rate = ((x_prob / (1-true_upcoding_rates[i])) - x_prob) / (1 - x_prob)
        prob_required_for_downcoding_rate = (((1 - x_prob) / (1-true_downcoding_rates[i])) - (1 - x_prob)) / x_prob

        df[variable_name] = (df[variable_name] + df['AGENT']*(1-df[variable_name]) * np.random.binomial(1, clamp(prob_required_for_upcoding_rate),  len(df))
                            - df[variable_name]*(1-df['AGENT']) * np.random.binomial(1, clamp(prob_required_for_downcoding_rate), len(df)))
        
        values = np.array([0.0, 0.0, 0.0, 0.05, 0.10, 0.15, 0.20])
        causal_effect = np.random.choice(values)
        causal_effect_of_x_on_y_list.append(causal_effect)
        y_prob += df[variable_name] * causal_effect
        
    # muskaan comment: te i.e., x1_causal_effect_on_y, is beta(x) in the paper
    df['Y'] = np.random.binomial(1, clamp(y_prob), len(df))

    return df, causal_effect_of_x_on_y_list

# Sensitivity Analysis Results On Semi-Synthetic Loan Data

### Vary Genuine Adaptation of A onto X2*

In [11]:
num_sims = 2
n = 10
# Dataframes to keep track of results
cmre_ds_control_for_nothing = pd.DataFrame(columns=['sim_num', 'sa', 'mr'])

num_correct = 0

# Perform a few simulations for each method
for sim in range(num_sims):
    # Set the random seed
    np.random.seed(sim)

    # Generate dataset for simulation
    df, causal_effect_of_x_on_y = generate_sgc_datasets(n, [0.1, 0.2, 0.3, 0.0, 0.05, 0.06, 0.07, 0.04, 0.03, 0.3, 0.4, 0.3, 0.1],
                    [0.04, 0.1, 0.2, 0.3, 0.0, 0.05, 0.06, 0.4, 0.03, 0.6, 0.1, 0.4, 0.3, 0.1],  
                    [ 0.04, 0.03, 0.6, 0.1, 0.2, 0.3, 0.0, 0.05, 0.01, 0.07, 0.4, 0.3, 0.1],
                    [ 0.3, 0.05, 0.06, 0.17, 0.03, 0.2, 0.1, 0.2, 0.3, 0.0, 0.4, 0.3, 0.1])
    
    p_values = kci_test(df, n, ['AGENT', 'EDUCATION', 'SEX', 'MARRIAGE', 'AGE'])

    for i in range(n):
        true_ce = causal_effect_of_x_on_y[i] > 0.01
        cond_dep = (p_values[i] < 0.05)
        if(true_ce == cond_dep):
            num_correct += 1

num_datapoints = num_sims * n
accuracy = num_correct / num_datapoints
print(f"accuracy is {accuracy}")

accuracy is 0.7
